# C3 checkup on Colab T4

Target: `max`, `t01`, `t11`. This is the ground-truth PII upper bound, not the OSINT condition.

Before running: upload `c3-check.zip` and `passllm_app.patch` when prompted.
The notebook patches `float16`, `MIN_PASSWORD_LENGTH=5`, `MAX_PASSWORD_LENGTH=24`, batch 16, and 10 inference runs for T4.

Do not use `t15` here. It is a random control; C3 on it should be zero by design.


In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.get_device_name(0))
print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
%cd /content
!rm -rf PassLLM
!git clone https://github.com/Tzohar/PassLLM.git
%cd PassLLM
!pip install -q -r requirements.txt


In [ ]:
!curl -L "https://github.com/Tzohar/PassLLM/releases/download/v1.3.0/PassLLM-Qwen3-4B-v1.0.pth" -o models/PassLLM_LoRA_Weights.pth
!ls -lh models/


In [ ]:
from google.colab import files
print("Upload c3-check.zip and passllm_app.patch")
files.upload()
!git apply passllm_app.patch
!unzip -q c3-check.zip
!ls c3-check/full


In [ ]:
from pathlib import Path
p = Path("src/config.py")
s = p.read_text()
repl = {
    'TORCH_DTYPE = "bfloat16"': 'TORCH_DTYPE = "float16"',
    "MIN_PASSWORD_LENGTH = 8": "MIN_PASSWORD_LENGTH = 5",
    "MAX_PASSWORD_LENGTH = 16": "MAX_PASSWORD_LENGTH = 24",
    "INFERENCE_BATCH_SIZE = 64": "INFERENCE_BATCH_SIZE = 16",
    "INFERENCE_NUM_RUNS = 50": "INFERENCE_NUM_RUNS = 10",
}
for old, new in repl.items():
    assert old in s, old
    s = s.replace(old, new)
p.write_text(s)
print("patched T4 config")
!grep -n "TORCH_DTYPE\|MIN_PASSWORD_LENGTH\|MAX_PASSWORD_LENGTH\|INFERENCE_BATCH_SIZE\|INFERENCE_NUM_RUNS" src/config.py


If you hit CUDA out of memory, replace `USE_4BIT = False` with `USE_4BIT = True` in `src/config.py`, restart the runtime, and rerun from the clone cell. Quality may shift slightly; report which path you used.

In [ ]:
import json
open("probe.jsonl", "w").write(json.dumps({"name": "Test Persona", "birth_year": "1990", "email": "probe@example.com"}) + "\n")
!python app.py --file probe.jsonl --weights models/PassLLM_LoRA_Weights.pth --fast


In [ ]:
import json
d = json.load(open("results/guesses_Test_Persona.json"))
pws = [x["password"] for x in d]
rank = pws.index("Test1990") + 1 if "Test1990" in pws else None
print("count", len(pws), "Test1990 rank", rank)
assert rank is not None, "Probe failed: stop C3 and inspect LoRA load and prompt keys."


In [ ]:
import glob, os, shutil, subprocess, sys
os.makedirs("c3-results/full", exist_ok=True)
os.makedirs("c3-results/projected", exist_ok=True)
for path in glob.glob("results/guesses_Test_Persona.json"):
    os.remove(path)

targets = ["max", "t01", "t11"]
for variant in ("full", "projected"):
    for t in targets:
        subprocess.run(
            [sys.executable, "app.py", "--file", f"c3-check/{variant}/{t}.jsonl",
             "--weights", "models/PassLLM_LoRA_Weights.pth", "--fast"],
            check=True,
        )
    for path in glob.glob("results/guesses_*.json"):
        shutil.copy(path, f"c3-results/{variant}/")
    print(variant, "done")


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("c3-results", "zip", "c3-results")
files.download("c3-results.zip")


## After download, score locally

```bash
cd ~/thesis-research/OSINT-AI-PWG
unzip -o ~/Downloads/c3-results.zip -d .
PYTHONPATH=src python scripts/check_c3.py --results c3-results --targets max t01 t11
```

Gates:

- `Test1990` missing from the probe means the engine is broken. Do not read C3 as an OSINT result.
- `t11` zero in both variants means name plus year is not being used; check weights and prompt.
- `max` full zero but projected hit means the 9-field prompt dilutes.
- `max` PassLLM zero but `expand=7` / `hybrid=7` means pattern coverage, not PII mapping.
- All zero means stop and inspect `load_state_dict(strict=False)` for skipped LoRA keys.
